In [ ]:
#@title 1. Setup Environment
import os, sys, json, time
import numpy as np
import torch

# Detect Environment (Colab vs Kaggle vs Local)
if 'google.colab' in sys.modules:
    ENV = 'colab'
    WORK_DIR = '/content'
elif os.path.exists('/kaggle'):
    ENV = 'kaggle'
    WORK_DIR = '/kaggle/working'
else:
    ENV = 'local'
    WORK_DIR = os.getcwd()

os.chdir(WORK_DIR)

# Clone ProteinMPNN
if not os.path.isdir("ProteinMPNN"):
    os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
sys.path.append(os.path.join(WORK_DIR, 'ProteinMPNN'))

print(f"✅ ProteinMPNN cloned and ready. Working directory: {WORK_DIR}")

✅ ProteinMPNN cloned and ready


In [ ]:
#@title 2. Load ProteinMPNN Model
import warnings
warnings.filterwarnings("ignore")

from protein_mpnn_utils import parse_PDB, ProteinMPNN, tied_featurize

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model settings - good balance of quality/speed
model_name = "v_48_020"  # @param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]
backbone_noise = 0.00

path_to_model_weights = os.path.join(WORK_DIR, 'ProteinMPNN/vanilla_model_weights')
checkpoint_path = f"{path_to_model_weights}/{model_name}.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)
model = ProteinMPNN(num_letters=21,
                    node_features=128,
                    edge_features=128,
                    hidden_dim=128,
                    num_encoder_layers=3,
                    num_decoder_layers=3,
                    augment_eps=backbone_noise,
                    k_neighbors=checkpoint['num_edges'])
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Model {model_name} loaded (noise level: {checkpoint['noise_level']}Å)")

Using device: cuda:0
✅ Model v_48_020 loaded (noise level: 0.2Å)


In [ ]:
#@title 3. Toxin Seed List + Helpers
import re
import numpy as np
from protein_mpnn_utils import parse_PDB, StructureDatasetPDB, _S_to_seq

toxin_list = [
    {"name": "Alpha-Bungarotoxin", "pdb": "1IK8", "chain": "A"},
    {"name": "Charybdotoxin", "pdb": "2CRD", "chain": "A"},
    {"name": "Agitoxin-2", "pdb": "1AGT", "chain": "A"},
    {"name": "Kappa-Bungarotoxin", "pdb": "1KBA", "chain": "A"},
]

def get_pdb(pdb_code):
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"

def compute_sequence_identity(seq1, seq2):
    matches = sum(a == b for a, b in zip(seq1, seq2))
    return (matches / max(len(seq1), len(seq2))) * 100 if max(len(seq1), len(seq2)) > 0 else 0

In [ ]:
#@title 4. 🚀 Improved Red-Teaming Loop (Better Output + Filtering)

# ================== CONFIG ==================
pdb_code = "1IK8"          # Change this
designed_chains = "A"
num_seq_per_target = 16    # Increase if you want
sampling_temp = 0.25
min_identity_threshold = 40  # Only save if below this
# ===========================================

pdb_path = get_pdb(pdb_code)
designed_chain_list = re.sub("[^A-Za-z]+", ",", designed_chains).split(",")
fixed_chain_list = []

chain_list = list(set(designed_chain_list + fixed_chain_list))

pdb_dict_list = parse_PDB(pdb_path, input_chain_list=chain_list)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)

chain_id_dict = {pdb_dict_list[0]['name']: (designed_chain_list, fixed_chain_list)}

print(f"🎯 Designing {pdb_code} ...")

fixed_positions_dict = omit_AA_dict = tied_positions_dict = pssm_dict = bias_by_res_dict = None
omit_AAs_np = np.array([False] * 21)
bias_AAs_np = np.zeros(21)

saved_count = 0

with torch.no_grad():
    for protein in dataset_valid:
        batch_clones = [protein] * 1

        X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
        visible_list_list, masked_list_list, masked_chain_length_list_list, \
        chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
        pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = \
            tied_featurize(batch_clones, device, chain_id_dict,
                          fixed_positions_dict, omit_AA_dict, tied_positions_dict,
                          pssm_dict, bias_by_res_dict)

        for temp in [sampling_temp]:
            print(f"\n=== Temperature = {temp} ===")
            for i in range(num_seq_per_target):
                randn = torch.randn(chain_M.shape, device=X.device)

                sample_dict = model.sample(
                    X, randn, S, chain_M, chain_encoding_all, residue_idx, mask=mask,
                    temperature=temp,
                    omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos,
                    omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef,
                    pssm_bias=pssm_bias,
                    pssm_multi=0.0,
                    pssm_log_odds_flag=False,
                    pssm_log_odds_mask=None,
                    pssm_bias_flag=False,
                    bias_by_res=bias_by_res_all
                )

                S_sample = sample_dict["S"][0].cpu().numpy()
                seq_str = _S_to_seq(S_sample, chain_M[0])

                original_seq = pdb_dict_list[0][f"seq_chain_{designed_chains}"]
                identity = compute_sequence_identity(original_seq, seq_str)

                print(f"Variant {i+1:02d} | Identity: {identity:.1f}%")

                if identity < min_identity_threshold:
                    with open(os.path.join(WORK_DIR, "designed_toxins.fasta"), "a") as f:
                        f.write(f">redesign_{pdb_code}_T{temp}_id{identity:.1f}_var{i}\n{seq_str}\n")
                    saved_count += 1

print(f"\n✅ Finished! Saved {saved_count} low-identity sequences to {os.path.join(WORK_DIR, 'designed_toxins.fasta')}")
# Kaggle UI provides access to output files natively in /kaggle/working
if ENV == 'colab':
    from google.colab import files
    files.download(os.path.join(WORK_DIR, "designed_toxins.fasta"))


🎯 Designing 1IK8 ...

=== Temperature = 0.25 ===
Variant 01 | Identity: 13.5%
Variant 02 | Identity: 20.3%
Variant 03 | Identity: 16.2%
Variant 04 | Identity: 18.9%
Variant 05 | Identity: 18.9%
Variant 06 | Identity: 17.6%
Variant 07 | Identity: 17.6%
Variant 08 | Identity: 14.9%
Variant 09 | Identity: 17.6%
Variant 10 | Identity: 17.6%
Variant 11 | Identity: 20.3%
Variant 12 | Identity: 20.3%
Variant 13 | Identity: 18.9%
Variant 14 | Identity: 23.0%
Variant 15 | Identity: 20.3%
Variant 16 | Identity: 18.9%

✅ Finished! Saved 16 low-identity sequences to designed_toxins.fasta


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title 5. 🚀 AUTO-FETCH 500 TOXIN PDBs & RUN "IDENTITY DROP" BATCH

!pip install -q biopython requests

import requests
import random
import os
from Bio import SeqIO
import numpy as np

# ================== CONFIG ==================
sampling_temps = [0.05, 0.15, 0.25, 0.30] 
num_seq_per_target_per_temp = 5  
random_seed = 42
TARGET_TOXIN_COUNT = 125 # 125 toxins * 4 temps * 1 variant (or 5 variants) = ~500 to 2500 total sequences
# ===========================================

print("🌐 Fetching real toxin PDBs from UniProt...")

# Query UniProt for Reviewed (Swiss-Prot) Toxins (KW-0800) that have a solved 3D structure (PDB)
uniprot_url = f"https://rest.uniprot.org/uniprotkb/search?query=(keyword:KW-0800)+AND+(database:pdb)+AND+(reviewed:true)&size=500&fields=accession,protein_name,xref_pdb"
response = requests.get(uniprot_url)
data = response.json()

fetched_toxins = []
for result in data.get('results', []):
    uniprot_id = result['primaryAccession']
    name = result.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', {}).get('value', f"Toxin_{uniprot_id}")
    
    # Extract the first PDB ID available for this toxin
    pdbs = [x['id'] for x in result.get('crossReferences', []) if x['database'] == 'PDB']
    if pdbs:
        fetched_toxins.append({
            "name": name.replace(" ", "_").replace("/", "").replace(",", ""),
            "uniprot": uniprot_id,
            "pdb": pdbs[0]  # Just use the first available 3D structure
        })

print(f"✅ Found {len(fetched_toxins)} unique toxins with PDB structures!")

# Shuffle and select our target count
random.seed(random_seed)
random.shuffle(fetched_toxins)
toxin_list = fetched_toxins[:TARGET_TOXIN_COUNT]

print(f"🎲 Selected {len(toxin_list)} toxins for the structural redesign pipeline.\n")

# ====================== Generation ======================
saved_total = 0
out_file = os.path.join(WORK_DIR, "identity_scale_toxin_batch.fasta")
open(out_file, "w").close()  # create empty file

with torch.no_grad():
    for toxin in toxin_list:
        print(f"\n🎯 Processing {toxin['name'][:30]}... ({toxin['pdb']})")
        
        try:
            pdb_path = get_pdb(toxin['pdb'])
            designed_chain_list = ["A"]
            fixed_chain_list = []
            pdb_dict_list = parse_PDB(pdb_path, input_chain_list=designed_chain_list)
            dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)
            chain_id_dict = {pdb_dict_list[0]['name']: (designed_chain_list, fixed_chain_list)}
        except Exception as e:
            print(f"   ⚠️ Error parsing structure {toxin['pdb']}, skipping...")
            continue

        fixed_positions_dict = omit_AA_dict = tied_positions_dict = pssm_dict = bias_by_res_dict = None
        omit_AAs_np = np.array([False] * 21)
        bias_AAs_np = np.zeros(21)

        toxin_saved = 0

        for protein in dataset_valid:
            batch_clones = [protein] * 1
            
            try:
                X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
                visible_list_list, masked_list_list, masked_chain_length_list_list, \
                chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
                pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = \
                    tied_featurize(batch_clones, device, chain_id_dict,
                                  fixed_positions_dict, omit_AA_dict, tied_positions_dict,
                                  pssm_dict, bias_by_res_dict)
            except Exception as e:
                print(f"   ⚠️ Featurization failed, skipping...")
                continue

            for temp in sampling_temps:
                for i in range(num_seq_per_target_per_temp):
                    randn = torch.randn(chain_M.shape, device=X.device)

                    sample_dict = model.sample(
                        X, randn, S, chain_M, chain_encoding_all, residue_idx, mask=mask,
                        temperature=temp,
                        omit_AAs_np=omit_AAs_np,
                        bias_AAs_np=bias_AAs_np,
                        chain_M_pos=chain_M_pos,
                        omit_AA_mask=omit_AA_mask,
                        pssm_coef=pssm_coef,
                        pssm_bias=pssm_bias,
                        pssm_multi=0.0,
                        pssm_log_odds_flag=False,
                        pssm_log_odds_mask=None,
                        pssm_bias_flag=False,
                        bias_by_res=bias_by_res_all
                    )

                    S_sample = sample_dict["S"][0].cpu().numpy()
                    seq_str = _S_to_seq(S_sample, chain_M[0])

                    original_seq = pdb_dict_list[0].get("seq_chain_A", "")
                    identity = compute_sequence_identity(original_seq, seq_str) if original_seq else 0.0

                    with open(out_file, "a") as f:
                        header = f">redesign_{toxin['uniprot']}_{toxin['pdb']}_T{temp}_id{identity:.1f}_var{i}"
                        f.write(f"{header}\n{seq_str}\n")
                    toxin_saved += 1
                    saved_total += 1

        print(f"   → Saved {toxin_saved} variants.")

print(f"\n🎉 BATCH COMPLETE! Total variants generated: {saved_total}")
print(f"File saved to: {out_file}")

if ENV == 'colab':
    from google.colab import files
    files.download(out_file)

🎲 Building batch with toxins that have PDB structures...
Selected 15 toxins with PDBs


🎯 Processing Kappa-Bungarotoxin (1KBA)
   Variant 01 | Identity: 40.9%
   Variant 02 | Identity: 37.9%
   Variant 03 | Identity: 37.9%
   Variant 04 | Identity: 36.4%
   Variant 05 | Identity: 43.9%
   Variant 06 | Identity: 37.9%
   Variant 07 | Identity: 37.9%
   Variant 08 | Identity: 40.9%
   → Saved 5 variants


🎯 Processing Alpha-Bungarotoxin (1IK8)
   Variant 01 | Identity: 20.3%
   Variant 02 | Identity: 20.3%
   Variant 03 | Identity: 17.6%
   Variant 04 | Identity: 20.3%
   Variant 05 | Identity: 21.6%
   Variant 06 | Identity: 17.6%
   Variant 07 | Identity: 14.9%
   Variant 08 | Identity: 21.6%
   → Saved 8 variants


🎯 Processing Charybdotoxin (2CRD)
   Variant 01 | Identity: 30.6%
   Variant 02 | Identity: 33.3%
   Variant 03 | Identity: 41.7%
   Variant 04 | Identity: 36.1%
   Variant 05 | Identity: 36.1%
   Variant 06 | Identity: 36.1%
   Variant 07 | Identity: 36.1%
   Variant 08 | 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title 6. 🔍 Quick Check on the Latest Generated Batch

from Bio import SeqIO

# Use the correct filename from the new Cell 5
input_fasta = out_file

print("=== First 5 Generated Sequences ===\n")
try:
    sequences = list(SeqIO.parse(input_fasta, "fasta"))
    
    for i, record in enumerate(sequences[:5]):
        seq = str(record.seq)
        print(f"{record.id}")
        print(f"{seq[:80]}{'...' if len(seq)>80 else ''}")
        print(f"Length: {len(seq)} aa\n")
    
    print(f"Total sequences in dataset: {len(sequences)}")
    print("\n✅ Ready for mapping the detection curve in Biosecurity_Hackathon.ipynb!")
    print(f"Download the '{os.path.basename(input_fasta)}' file from the working directory on the right-hand panel in Kaggle.")

except Exception as e:
    print(f"File not found or empty: {e}")

=== First 10 Generated Sequences ===

redesign_Kappa-Bungarotoxin_1KBA_T0.08_id37.9_var1
IKCYTTPNNTPVTCKPGETICYKICTCGPDCETTGCTCSRGCASSCPPKKPNYAGLLCCKGDLCNR
Length: 66 aa

redesign_Kappa-Bungarotoxin_1KBA_T0.08_id37.9_var2
IKCYTTPNNTPVTCKEGEDICYKICTCGPDCATTGCTCSRGCASSCPPKKPNDSGLLCCKGDLCNK
Length: 66 aa

redesign_Kappa-Bungarotoxin_1KBA_T0.08_id36.4_var3
IKCYTTPNNTVVTCKPGETICYRRETCGPDCATTGPTVSRGCVSSCPPPSPNDAGLQCCKGDLCNK
Length: 66 aa

redesign_Kappa-Bungarotoxin_1KBA_T0.08_id37.9_var5
IKCYTTPNNTPVTCKPGETICYKILKCGPNCETTGPTVSRGCASSCPPVQPNYVGLKCCKGDLCNR
Length: 66 aa

redesign_Kappa-Bungarotoxin_1KBA_T0.08_id37.9_var6
ITCYTTPNNKPVKCKPGEDICYKICTCGPNCATTGCTCSRGCASSCPPKKPNYAGLLCCKGDLCNK
Length: 66 aa

redesign_Alpha-Bungarotoxin_1IK8_T0.08_id20.3_var0
AACTGAGGGPAGGTGGPAGGGGCYCSGGAPGAAGGGGTGWGCGGSGACPGGTAGGVSGCPGGGSGCPAPGAAPG
Length: 74 aa

redesign_Alpha-Bungarotoxin_1IK8_T0.08_id20.3_var1
AACAGAGGGPSGGAGGPGGGGGCACAGGAPGGAGGGGAGPTCGGSGACPGGAPGPAAGCPGGGGGCPAPGGSPG
Length: 74 aa

redesign_Alpha-